# Student Performance Analysis and Prediction System

**Minor Project | AI/ML with Python Training Program**  
**Prepared by:** Shivam A Pandya  
**Program:** B.Tech - AI Engineering, SRM Ramapuram

This notebook follows the project workflow: inspect and clean the data, perform exploratory analysis, plot academic patterns, train Linear Regression, and evaluate it on an unseen test partition.

**Dataset note:** This project uses selected fields from the UCI Student Performance Portuguese-course dataset (649 records). `studytime` is a weekly band and `absences` is an absence count; neither is converted into an invented exact-hours or attendance-percentage field. See `data/source_info.md` for attribution and full variable notes.

## 1. Import libraries and set project paths

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "model.py").exists():
    # This fallback supports launching the notebook from a different working directory.
    PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from model import load_and_clean_data, save_eda, train_and_evaluate, ensure_directories

DATA_PATH = PROJECT_ROOT / "data" / "student_performance_dataset.csv"
print("Project folder:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists())

Project folder: /mnt/data/Student_Performance_Analysis_UCI
Dataset exists: True


## 2. Load and inspect the dataset

In [2]:
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn types:")
display(df.dtypes.to_frame("data_type"))
print("\nMissing values:")
display(df.isna().sum().to_frame("missing_count"))

Dataset shape: (649, 5)


,studytime,absences,G1,G2,G3
0,2,4,0,11,11
1,2,2,9,11,11
2,2,6,12,13,12
3,3,0,14,14,14
4,2,0,11,13,13



Column types:


,data_type
studytime,int64
absences,int64
G1,int64
G2,int64
G3,int64



Missing values:


,missing_count
studytime,0
absences,0
G1,0
G2,0
G3,0


## 3. Data cleaning and feature engineering

The model uses `studytime`, `absences`, `G1` and `G2` to estimate `G3`. Two descriptive groups (`studytime_band` and `absence_group`) are derived for EDA. The complete source file was checked for exact duplicates: none were found. Repeated combinations in the five selected columns are not deleted because different students can share the same selected values.

In [3]:
data, cleaning_summary = load_and_clean_data(DATA_PATH)
print(json.dumps(cleaning_summary, indent=2))
display(data.head())

{
  "rows_before_cleaning": 649,
  "rows_after_cleaning": 649,
  "exact_duplicate_rows_found_in_full_33_column_source": 0,
  "repeated_combinations_in_selected_working_columns": 109,
  "repeated_combinations_removed": 0,
  "missing_values_in_required_columns_before_cleaning": 0,
  "rows_removed_for_missing_target": 0,
  "feature_missing_values_imputed_by_pipeline": 0,
  "rows_removed_as_exact_duplicates": 0,
  "features": [
    "studytime",
    "absences",
    "G1",
    "G2"
  ],
  "target": "G3"
}


,studytime,absences,G1,G2,G3,studytime_band,absence_group
0,2,4,0,11,11,2-5 hours/week,4-10 absences
1,2,2,9,11,11,2-5 hours/week,0-3 absences
2,2,6,12,13,12,2-5 hours/week,4-10 absences
3,3,0,14,14,14,5-10 hours/week,0-3 absences
4,2,0,11,13,13,2-5 hours/week,0-3 absences


## 4. Descriptive statistics

In [4]:
summary = data[["studytime", "absences", "G1", "G2", "G3"]].describe().T
summary["median"] = data[["studytime", "absences", "G1", "G2", "G3"]].median()
display(summary.round(3))
print(f"Mean G3: {data['G3'].mean():.3f}")
print(f"Median G3: {data['G3'].median():.3f}")
print(f"Standard deviation of G3: {data['G3'].std():.3f}")

,count,mean,std,min,25%,50%,75%,max,median
studytime,649.0,1.931,0.830,1.0,1.0,2.0,2.0,4.0,2.0
absences,649.0,3.659,4.641,0.0,0.0,2.0,6.0,32.0,2.0
G1,649.0,11.399,2.745,0.0,10.0,11.0,13.0,19.0,11.0
G2,649.0,11.570,2.914,0.0,10.0,11.0,13.0,19.0,11.0
G3,649.0,11.906,3.231,0.0,10.0,12.0,14.0,19.0,12.0


Mean G3: 11.906
Median G3: 12.000
Standard deviation of G3: 3.231


## 5. Explore performance groups and relationships

In [5]:
eda_summary = save_eda(data)
print("Correlation with G3:")
display(pd.Series(eda_summary["correlations_with_G3"], name="correlation").to_frame())
print("\nMean final grade by weekly study-time band:")
display(pd.DataFrame(eda_summary["studytime_summary"]).round(3))
print("\nMean final grade by absence group:")
display(pd.DataFrame(eda_summary["absence_summary"]).round(3))

Correlation with G3:


,correlation
G2,0.918548
G1,0.826387
studytime,0.249789
absences,-0.091379



Mean final grade by weekly study-time band:


,studytime_band,count,mean,median,std
0,2-5 hours/week,305,12.092,12.0,3.243
1,5-10 hours/week,97,13.227,13.0,2.502
2,<2 hours/week,212,10.844,11.0,3.219
3,>10 hours/week,35,13.057,13.0,3.038



Mean final grade by absence group:


,absence_group,count,mean,median,std
0,0-3 absences,373,12.067,12.0,3.635
1,4-10 absences,227,11.912,11.0,2.558
2,More than 10,49,10.653,10.0,2.429


## 6. Required visualizations

The script saves a bar chart, histogram, three scatter plots, a box plot and a correlation heatmap in `outputs/visualizations/`.

In [6]:
from IPython.display import display, Image
plot_dir = PROJECT_ROOT / "outputs" / "visualizations"
plot_files = [
    "bar_mean_grade_by_studytime.png",
    "histogram_final_grade.png",
    "scatter_studytime_vs_final_grade.png",
    "scatter_absences_vs_final_grade.png",
    "scatter_g2_vs_final_grade.png",
    "boxplot_grade_by_studytime.png",
    "correlation_heatmap.png",
]
for filename in plot_files:
    print(f"Saved visualization: outputs/visualizations/{filename}")

Saved visualization: outputs/visualizations/bar_mean_grade_by_studytime.png
Saved visualization: outputs/visualizations/histogram_final_grade.png
Saved visualization: outputs/visualizations/scatter_studytime_vs_final_grade.png
Saved visualization: outputs/visualizations/scatter_absences_vs_final_grade.png
Saved visualization: outputs/visualizations/scatter_g2_vs_final_grade.png
Saved visualization: outputs/visualizations/boxplot_grade_by_studytime.png
Saved visualization: outputs/visualizations/correlation_heatmap.png


## 7. Train the Linear Regression model

An 80/20 train-test split is used. Imputation and scaling are inside a Scikit-learn pipeline, so those transformations are fitted only on training folds. A small grid search compares the intercept setting using 5-fold cross-validation.

In [7]:
metrics = train_and_evaluate(data)
print(json.dumps(metrics, indent=2))

{
  "model": "Linear Regression",
  "input_features": [
    "studytime",
    "absences",
    "G1",
    "G2"
  ],
  "target": "G3",
  "train_rows": 519,
  "test_rows": 130,
  "random_state": 42,
  "test_MAE": 0.7477392675349168,
  "test_RMSE": 1.1713573177973222,
  "test_R2": 0.8592988178401071,
  "cross_validation_MAE_mean": 0.8162519083169416,
  "cross_validation_MAE_std": 0.107291461792493,
  "cross_validation_R2_mean": 0.8406963155572083,
  "best_parameters": {
    "regression__fit_intercept": true
  },
  "best_cv_MAE": 0.8162519083169416
}


## 8. Model evaluation and predictions

In [8]:
results_dir = PROJECT_ROOT / "outputs" / "evaluation_results"
predictions = pd.read_csv(results_dir / "test_predictions.csv")
print("Evaluation metrics:")
display(pd.Series({
    "MAE": metrics["test_MAE"],
    "RMSE": metrics["test_RMSE"],
    "R2": metrics["test_R2"],
    "5-fold CV MAE mean": metrics["cross_validation_MAE_mean"],
    "5-fold CV MAE standard deviation": metrics["cross_validation_MAE_std"],
    "5-fold CV R2 mean": metrics["cross_validation_R2_mean"],
}).to_frame("value").round(3))
print("\nSample actual and predicted G3 values:")
display(predictions.head(12).round(2))

Evaluation metrics:


,value
MAE,0.748
RMSE,1.171
R2,0.859
5-fold CV MAE mean,0.816
5-fold CV MAE standard deviation,0.107
5-fold CV R2 mean,0.841



Sample actual and predicted G3 values:


,test_row_index,actual_G3,predicted_G3,absolute_error
0,2,12,13.36,1.36
1,6,13,12.52,0.48
2,10,14,14.52,0.52
3,24,10,11.25,1.25
4,30,11,11.08,0.08
5,31,15,15.58,0.58
6,39,12,13.66,1.66
7,41,11,11.15,0.15
8,44,11,11.42,0.42
9,49,12,12.57,0.57


## 9. Interpretation, limitations and conclusion

In this run, G2 has the strongest linear relationship with G3 (correlation approximately 0.919), followed by G1 (approximately 0.826). Study-time category has a smaller positive relationship (approximately 0.250), while absences show a weak negative relationship (approximately -0.091). On the held-out test set, MAE is approximately 0.748, RMSE is approximately 1.171 and R-squared is approximately 0.859.

These are associations from a dataset covering two Portuguese schools, not causal effects or universal findings. The strongest predictors are earlier period grades, so this model is appropriate only after G1 and G2 are available. `studytime` is a weekly category and `absences` is a count rather than attendance percentage. Predictions should be treated as estimates to support review, not as a decision about a student.